# FinSight AI — Research Notebook 02: Feature Engineering & Leakage Prevention
## Constructing Rolling Indicators, Lags, and Enforcing Chronological Train/Test Split Boundaries

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import sys
sys.path.append('../backend')

from app.ml.feature_engineering import compute_features, create_target, time_series_split, FEATURE_COLUMNS

In [ ]:
# Ingest OHLCV
raw = yf.download("AAPL", period="5y", interval="1d")
raw = raw[["Open", "High", "Low", "Close", "Volume"]].dropna()
print(f"Raw observations: {len(raw)}")

In [ ]:
# Generate Technical Features
feat_df = compute_features(raw)
print(f"Engineered Features Count: {len(FEATURE_COLUMNS)}")
feat_df[FEATURE_COLUMNS].head()

In [ ]:
# Construct Target & Verify Chronological Order (No Look-Ahead)
X, y = create_target(feat_df, horizon=5)
X_train, X_val, X_test, y_train, y_val, y_test = time_series_split(X, y)

print(f"Train range: {X_train.index[0].date()} to {X_train.index[-1].date()} ({len(X_train)} samples)")
print(f"Validation range: {X_val.index[0].date()} to {X_val.index[-1].date()} ({len(X_val)} samples)")
print(f"Test range: {X_test.index[0].date()} to {X_test.index[-1].date()} ({len(X_test)} samples)")

assert X_train.index[-1] < X_val.index[0] < X_test.index[0], "ERROR: Time overlap detected!"